In [1]:
## This program creates the precursors for localized energy consumption videos and the graph of total consumption within the gel.
## I don't know if I have to import the relevant libraries in this program or in the master program. 

# From Strain rate code
# To handle images, we have PIL - Image
import numpy as np  # Essential functions
#import numpy.ma as ma # For masking, removed temporarily
import pandas as pd # Essential for dataframe and excel manipulation
import matplotlib.pyplot as plt # Essential data visualization
#from matplotlib import cm # Creates colorbar, removed temporarily
#from matplotlib import colors # Colors for plots, removed temporarily

# Collection of image processing directories.
from skimage.filters import threshold_otsu #The binarization algorithm
from skimage.io import imread #Image processing
#from skimage.viewer import ImageViewer #Probably defunct, removed temporarily
from skimage.morphology import reconstruction # For floodfill algorithm
%matplotlib inline
#from sklearn.linear_model import LinearRegression # Removed temporarily
import os.path as ospath # os.path used to check if the csv exists.
# from mpl_toolkits.axes_grid1 import make_axes_locatable, # Removed temporarily
import os as os



In [2]:
######################################################################### Not Defunct

# Essential path names and opening terms
# This block assigns the directory we work in, and the filenames we generate and use.

# Specify the overarching path for our filenames.
path = "C:\\Users\\franc\\Box\\ALAB Data Francis Cavanna\\2021 Actomyosin Control\\"

#Specify DAPI filename base:
base_name = 'Actin_12uM_Fascin_1,2uM_Myosin_0,12uM-Create Image Subset and Split-02'

#Specify PLUM filename base:
PLUM_base = 'StrainRateFiles\\Actin_12uM_Fascin_1,2uM_Myosin_0,12uM-Create Image Subset and Split-01'

# Specify base of the filenames of the series of images:
DAPI_base = 'StrainRateFiles\\' + base_name

folder = 'StrainRateFiles\\EnergyConsumpVideos\\' + base_name

other_folder = 'StrainRateFiles\\Energy_Difference\\' + base_name

# Specify suffix of the filenames of the series of images
suffix = '.tif'

# Specify the Energy Consump6tion Excel Document name:
ECfilename = "EnergyConsumption.csv"

# Specify the Strain Rate Excel Document name:
SRfilename = "StrainRate.csv"

# Specify originator Excel Document from MolarConcentration.nb program:
MCfilename = "MathematicaDF.csv"

ATPFreeEnergy = 56 # Gives kJ/Mol energy consumption, or J/mMol energy consumption.
#ATPFreeEnergy = 58 # Gives the better kJ/Mol energy consumption, or J/mMol energy consumption value, after interpolation.

# Specify truncation term:
truncate = 0

# Import the normalization curve:
df_norm = pd.read_csv("ExcelValuesNADHControl1.csv")

# Load series of TIFF images for PLUM and DAPI

# Specify number of slices
N_slices = 91

# Collect an array of filenames to load
filenames_PLUM = []
filenames_DAPI = []
PLUM_savenames = []
DAPI_savenames = []
for n in np.arange(1,N_slices):
    
    # Convert integer to 0001 format
    number04d = format("%04d"% (n))
    
    # Connect base, format, and suffix to make a filename
    #filename = base + suffix
    filename_PLUM = PLUM_base + number04d + suffix
    filename_DAPI = DAPI_base + number04d + suffix
    PLUM_savename = folder + number04d
    DAPI_savename = other_folder + number04d
    
    # Append to filenames
    filenames_PLUM.append(filename_PLUM)
    filenames_DAPI.append(filename_DAPI)
    PLUM_savenames.append(PLUM_savename)
    DAPI_savenames.append(DAPI_savename)

######################################################################### Not Defunct    



In [3]:
    
# Use filenames to load PLUM images
def Load_PLUM(filenames_PLUM):
        images_PLUM = [] #HI!
        for fn in filenames_PLUM:

            # Load image
            im = imread(fn)

            images_PLUM.append(im)
        return images_PLUM

In [ ]:
# Use filenames to load DAPI images
def Load_DAPI(filenames_DAPI):
    images_DAPI = []
    for fn in filenames_DAPI:

        # Load image
        im = imread(fn)

        images_DAPI.append(im)
    return images_DAPI

In [ ]:
# Threshold additional images to eliminate outliers in center of actin mesh
def Thresholding(fn):
    # Load image
    im_PLUM = imread(fn)

    threshold = threshold_otsu(im_PLUM)
    thresholded = im_PLUM > threshold

    # Modify the datatype to prevent a "conversion from float to uint8" error.
    seed = np.ones_like(thresholded, dtype=np.uint8)*4095.0
    
    #Insert floodfill algorithm here:
    thresholded[ : ,0] = 0.0 #Sets all values on the left side of the image to 0.
    thresholded[ : ,-1] = 0.0 #Sets all values on the right side of the image to 0.
    thresholded[ 0 ,:] = 0.0 #Sets all values on the top of the image to 0
    thresholded[ -1 ,:] = 0.0 #Sets all values on the bottom of the image to 0.
    seed[ : ,0] = 0.0 #See above.
    seed[ : ,-1] = 0.0
    seed[ 0 ,:] = 0.0
    seed[ -1 ,:] = 0.0

    # Require every "dark" pixel that's not contiguous with the edge to become bright.
    fill = reconstruction(seed, thresholded, method='erosion')
    #print(fill)
    return fill

In [ ]:
####################   PROVISIONAL CODE FOR TESTING ARRAY MODIFICATION ######################

# Create mask for vacuum grease to be eliminated
def Grease_Mask(fn):
    # Load image
    # Modify datatype to prevent a "ValueError: cannot convert float NaN to integer"
    im_DAPI = imread(fn).astype(dtype=np.float64)
    
    # Identify regions that hit the pixel ceiling
    # Pixel ceiling range in file is 4095
    Grease_Locations = (im_DAPI < 4095)
    
    # Mask original image to eliminate values covered by grease
    im_DAPI[Grease_Locations] = np.nan
    # Return Array
    return im_DAPI
    
    
#################################################################################################

#plt.imshow(Grease_Mask(filenames_DAPI[0]),cmap='viridis')

In [ ]:
### Create Circular mask to find total energy consumption

def Circ_Mask(List_Loc,Diff,t):
    #List_Loc is a list of points within the array (x,y) that give values derived from the mask pulled from actin mesh
    r = np.sqrt(Diff*t/np.pi)
    List = []
    for loc in List_Loc:
        for i in range(0,r):
            for j in range(0,r):
                if (i**2 + j**2 < r**2):
                    if (x-i) and (y-j) >= 0:
                        List.append((x-i,y-j))
                    if (x+i) and (y+j) <= 2048:
                        List.append((x+i,y+j))
    List = set(List+List_Loc)
    return List

In [ ]:
# Save Images as a png to be processed into videos

def Find_Energy(folder,DAPI_Files,C1images):
    images_DAPI = Load_DAPI(DAPI_Files)
    
    # Fits from Mathematica MolarConcentration_Full_NADH_Calibration.nb document.
    Fit1 = -0.12123625148063022
    Fit2 = 0.00046298638444968403

    # Find the maximum pixel value to rescale our images
    fim = images_DAPI[-1]
    # Apply intensity-to-energy conversion from NADH calibration curve. Returns mM*J/mmol = J/L measurement
    fim = (Fit1 + Fit2*fim)*ATPFreeEnergy
    Starter = (Fit1 + Fit2*images_DAPI[0])*ATPFreeEnergy
    Base = (Fit1 + Fit2*df_norm["Mean"])*ATPFreeEnergy
    Ceiling = np.max(fim)

    #Create arrays for the sum of energy consumption and the average energy consumption
    e_sum = []
    e_avg = []
    En_sum = 0

    for i in range(1,len(images_DAPI)-1):
        # Pull out images and names
        im = images_DAPI[i]
        fn = DAPI_Files[i]
        C1_arr = C1images[i]
        C1_arr_next = C1images[i+1]

        # Convert intensity to energy consumption in each image and calculate the total energy consumption.
        # The units of this fit is mM*J/mmol = J/L.
        im = (Fit1 + Fit2*im)*ATPFreeEnergy
        im_next = (Fit1 + Fit2*images_DAPI[i+1])*ATPFreeEnergy
        saveData = Starter - im + Base[i] - Base[0]      # difference between initial fluoresence and current, 
                                                        # minus difference between calibration curve and 
        saveData_next = Starter - im_next + Base[i+1] - Base[0]
        
        #Create a mask for the drops of NADH on exterior of the slide.
        rmv_image = np.where(im==0, 0, 1)
        
        # Get desired energy consumption for images
        e = np.multiply(saveData,C1_arr)
        g = np.multiply(e,rmv_image)
        e_next = np.multiply(saveData_next,C1_arr_next)
        g_next = np.multiply(e_next,rmv_image)

        #Conditional code for eliminating 0's in array:
        g[g==0] = np.nan
        g_next[g_next==0] = np.nan
        
        #e is the array of energy-per-volume values.
        e_avg.append((np.nanmean(g_next)-np.nanmean(g))/20) # Saved to csv. This variable is in J/L/s, gives 
                                                            # the euler approximation for Energy Consumption.

        En_sum = np.nanmean(g)# Sums units that are not zero.
        
        #En_sum = En_sum + np.sum(e)/len(e)#*((0.7692**2)*150.0/(10000.0**3)) Orphaned code that calculates total Joules.
        
        e_sum.append(En_sum) # Saved to csv. This variable is in J/L, gives the cumulative energy consumption.

        # Save the image as an png.
        image_color = plt.imshow(saveData,cmap='viridis')
        cb = plt.colorbar(image_color)
        plt.clim(0,Ceiling)
        
        plt.savefig(fn + ".png")
        # Remove and clear the image to allow clean processing of the next images.
        cb.remove()
        plt.clf()

    # We save data in a Joules/L, Joules/L/s setup
    Data = [e_sum,e_avg]
    if ospath.exists(path + folder + '\Results//Strains.csv') is True:
        os.remove(path + folder + '\Results//Energies.csv')
    else:
        pass
    df = pd.DataFrame(Data,index=["g (J/L)","g_dot (J/L/s)"])
    # save to collect strain rates from the csv.
    df.to_csv(path + folder + '\Results//Energies.csv', index=True, mode='a',header=False)

    
#This pulls out only energy difference in J/mL, unless material is appropriately converted

# Preliminary dimensional parameters, pending additional experiments

#width per pixel: 0.7692 pixel/1.0 micron
#Height per pixel: 0.7692 pixel/1.0 micron

#Depth of slide: 150 microns

#Per-pixel volume: 0.7692*0.7692*150 microns^3

#Total in Joules: J/ml*1mL/cm^3*1cm^3/1000^3 um^3*0.7692*0.7692*150 microns^3

# Note to self: 1 cm = 10,000 um, not 1000 um!
